# SPLADE-Distil: эксперимент (MS MARCO)

Дистилляция из SPLADE v2 (arXiv:2109.10086): студент **SPLADE-max** учится у кросс-энкодера
**cross-encoder/ms-marco-MiniLM-L6-v2** по MarginMSE. Общий код — в `splade_lab/` (тот же, что
у `splade_experiments.ipynb`; логика прошлых моделей не меняется), новый модуль — `splade_lab/distil.py`.

Конвейер на одном GPU — две модели в памяти одновременно не живут:
1. **Учитель** считает margins по всем триплетам → кеш на диск, затем выгружается.
2. **Студент** (SPLADE-max) обучается на готовых margins (MarginMSE + FLOPS-рег.).
3. **Оценка** retrieval, артефакты в `outputs/<version>/<run_id>/`.

Порядок — как в `splade_experiments.ipynb`: клацай клетки сверху вниз.
`MODE = "smoke"` — маленький детерминированный сабсет, минуты; `"full"` — полный MS MARCO (часы на A100).

## 1. Setup

In [1]:
# Первый запуск на новой машине (в активированном venv) — раскомментировать:
%pip install -r requirements.txt
%load_ext autoreload
%autoreload 2

import json
import torch

from splade_lab.config import merge_config
from splade_lab.train import pick_device

device = pick_device()
print(f"torch {torch.__version__} | устройство: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

Looking in indexes: https://pypi.yandex-team.ru/simple/
Note: you may need to restart the kernel to use updated packages.
torch 2.5.1+cu124 | устройство: cuda (NVIDIA A100 80GB PCIe)


## 2. Конфиг данных

Тот же каталог `data/msmarco/<mode>/`, что и в `splade_experiments.ipynb` — данные
переиспользуются (если уже скачаны, повторно не качаются). Margins учителя кешируются
рядом с триплетами и пересчитываются только если их там нет.

In [2]:
MODE = "full"  # "smoke" | "full"  (full — гигабайты и часы, см. предупреждение при подготовке)

DATA = {
    # Официальные файлы Microsoft. Если хост недоступен — зеркало:
    # https://msmarco.blob.core.windows.net/msmarcoranking/
    "urls": {
        "triples": "https://msmarco.z22.web.core.windows.net/msmarcoranking/triples.train.small.tar.gz",
        "collection": "https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz",
        "queries": "https://msmarco.z22.web.core.windows.net/msmarcoranking/queries.tar.gz",
        "qrels_dev": "https://msmarco.z22.web.core.windows.net/msmarcoranking/qrels.dev.small.tsv",
    },
    "data_dir": "data/msmarco",  # относительно корня репозитория
    "smoke": {
        "num_train_triples": 256,  # строк triples на обучение
        "num_eval_queries": 20,    # уникальных eval-запросов (не пересекаются с train)
        "num_corpus_docs": 1000,   # корпус: positives eval-запросов + дистракторы
    },
    "full": {
        "num_train_triples": 3_200_000,  # ~ max_steps * batch_size; -1 = весь файл (39.8M строк)
        "num_eval_queries": -1,          # -1 = все 6980 запросов dev small
    },
}

print(f"MODE = {MODE} | параметры данных: {DATA[MODE]}")

MODE = full | параметры данных: {'num_train_triples': 3200000, 'num_eval_queries': -1}


## 3. Конфиг эксперимента

**DistilSPLADE-max** (SPLADE v2): общий MLM-энкодер для запросов и документов, обучение по
MarginMSE с учителем-кросс-энкодером. Гиперпараметры студента — как у `splade_experiments.ipynb`
(статья: distilbert, Adam 2e-5, warmup 6000, max_len 256). Новая секция `distil` задаёт учителя.

Новая версия = новая запись в `EXPERIMENTS` (переопределения поверх `BASE`). Код не трогать.

In [ ]:
# База: гиперпараметры статьи (distilbert, Adam 2e-5, warmup 6000, max_len 256).
# Отличие от статьи: 100k шагов и batch 124 (1xA100 против 4xV100 / batch 124 / 150k).
BASE = {
    "model": {
        "hf_model": "distilbert-base-uncased",
        "query_encoder": "mlm",      # mlm = SPLADE-max (студент дистилляции)
        "max_len_query": 32,
        "max_len_doc": 256,
    },
    "train": {
        "seed": 4,
        "lr": 2e-5,
        "batch_size": 124,
        "max_steps": 100_000,
        "warmup_steps": 6_000,
        "flops_warmup_steps": 10_000,  # квадратичный разгон лямбд (SPLADE v2)
        "lambda_q": 0.5,              # FLOPS-рег. запросов
        "lambda_d": 0.5,              # FLOPS-рег. документов
        "log_every": 100,
    },
    "eval": {
        "batch_size_docs": 256,
        "batch_size_queries": 64,
        "batch_size_search": 64,
        "recall_ks": [10, 100, 1000],
    },
    "distil": {
        "teacher": "cross-encoder/ms-marco-MiniLM-L6-v2",  # кросс-энкодер-учитель
        "max_len": 256,       # длина пары query+doc для учителя
        "batch_size": 128,    # размер батча учителя при подсчёте margins
    },
}

# Smoke: всё маленькое, схема артефактов та же.
SMOKE_OVERRIDES = {
    "model": {"max_len_doc": 128},
    "train": {"max_steps": 20, "batch_size": 8, "warmup_steps": 4,
              "flops_warmup_steps": 8, "log_every": 5},
    "eval": {"batch_size_docs": 32, "batch_size_queries": 32, "recall_ks": [10, 100]},
    "distil": {"max_len": 128, "batch_size": 16},
}

# Версии эксперимента: имя -> переопределения поверх BASE.
EXPERIMENTS = {
    "v3_splade_distil": {},
}

def build_config(name: str) -> dict:
    cfg = merge_config(BASE, EXPERIMENTS[name])
    if MODE == "smoke":
        cfg = merge_config(cfg, SMOKE_OVERRIDES)
    cfg["version"], cfg["mode"] = name, MODE
    return cfg

for name in EXPERIMENTS:
    print(f"=== {name} (mode={MODE}) ===")
    print(json.dumps(build_config(name), indent=2, ensure_ascii=False))

=== v3_splade_distil (mode=full) ===
{
  "model": {
    "hf_model": "distilbert-base-uncased",
    "query_encoder": "mlm",
    "max_len_query": 32,
    "max_len_doc": 256
  },
  "train": {
    "seed": 4,
    "lr": 2e-05,
    "batch_size": 124,
    "max_steps": 100000,
    "warmup_steps": 6000,
    "flops_warmup_steps": 10000,
    "lambda_q": 0.0003,
    "lambda_d": 0.0001,
    "log_every": 100
  },
  "eval": {
    "batch_size_docs": 256,
    "batch_size_queries": 64,
    "batch_size_search": 64,
    "recall_ks": [
      10,
      100,
      1000
    ]
  },
  "distil": {
    "teacher": "cross-encoder/ms-marco-MiniLM-L6-v2",
    "max_len": 256,
    "batch_size": 128
  },
  "version": "v3_splade_distil",
  "mode": "full"
}


## 4. Данные (скачиваются один раз)

In [4]:
from splade_lab import data as msdata

prepare = msdata.prepare_full if MODE == "full" else msdata.prepare_smoke
ds_dir = prepare(DATA)

print(f"\nКаталог: {ds_dir}")
print("Строк в файлах:", msdata.dataset_stats(ds_dir))

with open(ds_dir / "queries.tsv", encoding="utf-8") as f:
    print("\nПример запроса:  ", f.readline().strip()[:120])
with open(ds_dir / "collection.tsv", encoding="utf-8") as f:
    print("Пример пассажа:  ", f.readline().strip()[:120], "...")

[data] full уже готов: /home/uvuv/workspace/sandbox_03/data/msmarco/full

Каталог: /home/uvuv/workspace/sandbox_03/data/msmarco/full
Строк в файлах: {'collection.tsv': 8841823, 'queries.tsv': 6980, 'qrels.tsv': 7437, 'triples.tsv': 3200000}

Пример запроса:   2	 Androgen receptor define
Пример пассажа:   0	The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as s ...


## 5. Прогон (учитель → студент → оценка)

`run_experiment_distil`: этап 1 — учитель считает margins (кеш на диск, потом выгружается из
памяти); этап 2 — обучение студента по MarginMSE; этап 3 — eval. На GPU в каждый момент —
только одна модель. Прогресс и память (gpu/ram) пишутся в файл лога (путь печатается ниже).

In [ ]:
from splade_lab.distil import run_experiment_distil

results = {}
for name in EXPERIMENTS:
    cfg = build_config(name)
    run_dir, metrics = run_experiment_distil(cfg, DATA)
    results[name] = {"run_dir": str(run_dir), **metrics}

for name, res in results.items():
    print(f"{name}: mrr@10={res.get('mrr@10'):.4f} | {res['run_dir']}")

## 6. Сравнение версий (все прогоны из outputs/)

Та же таблица, что и в `splade_experiments.ipynb`: distil-прогон встанет рядом с базовыми
SPLADE-max / SPLADE-doc, если они уже прогонялись.

In [ ]:
from splade_lab.compare import compare_runs

df = compare_runs()  # печатает таблицу и пишет outputs/comparison.csv
df

## Как менять

**Другой учитель / регуляризация:** правь `BASE["distil"]["teacher"]` или `lambda_d` в клетке 3.
При смене учителя margins пересчитаются (новый кеш-файл рядом с триплетами).

**Новая версия:** добавь запись в `EXPERIMENTS`, например
`"v3b_distil_highreg": {"train": {"lambda_q": 6e-2, "lambda_d": 2e-2}}`, и перезапусти клетки 3 → 5 → 6.

**Full-прогон:** `MODE = "full"` в клетке 2, затем клетки 2 → 6 по порядку. Ресурсы: ~1GB
трафика + ~6GB диска на данные; учитель по 3.2M триплетов — заметное время (один раз, потом кеш);
обучение 100k шагов — часы на A100; кодирование 8.8M пассажей и поиск — как в базовом ноутбуке.